# How Many People in the World Can Squat 315 / 405 / 495?
### Two-pool model: gym squatters + non-gym strength athletes

> The bench press is almost entirely a gym movement — you need a bench and a barbell.  
> The squat is different. It's a fundamental human pattern.  
> Military recruits, combat sports athletes, rugby players, construction workers —  
> many have the leg strength to squat heavy without ever touching a squat rack.  
> This model accounts for both pools.

---
**Method:**  
OPL data (3.9M records) → fit squat distribution → adjust for gym-goer gap  
→ add non-gym strength athlete pool (military, combat sports, team sports S&C)  
→ 10-region world model → 100,000 Monte Carlo trials  
→ denominator analysis: all humans / able adults / men only / women only

---
## ⚙️ Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import lognorm as _lognorm
import warnings, requests, zipfile, io, os
warnings.filterwarnings('ignore')
np.random.seed(42)

BG    = '#0d0d0d'
PANEL = '#141414'
GRID  = '#252525'
WHITE = '#f0f0f0'
GRAY  = '#666666'
C315  = '#00b4ff'
C405  = '#ff2d2d'
C495  = '#ffd700'
MALE  = '#4fc3f7'
FEM   = '#f48fb1'
POOL_A_COLOR = '#4fc3f7'   # gym squatters
POOL_B_COLOR = '#ff9800'   # non-gym strength athletes

MILESTONES = [315, 405, 495]
M_COLORS   = [C315, C405, C495]
M_TAGS     = ['315 lbs', '405 lbs', '495 lbs']
M_LABELS   = ['315 lbs\n(3 plates)', '405 lbs\n(4 plates)', '495 lbs\n(5 plates)']
FRAC_MALE  = 0.504
WORLD_POP  = 8_200_000_000

# Denominator reference populations (UN 2024)
POP_ABLE_1670  = 4_674_000_000   # world 16-70
POP_MEN_1670   = 2_372_000_000   # men 16-70
POP_WOMEN_1670 = 2_302_000_000   # women 16-70

REEL_W, REEL_H = 6.75, 12.0
REEL_DPI = 160

def make_fig(w=13, h=6, cols=1):
    if cols == 1:
        fig, ax = plt.subplots(figsize=(w, h), facecolor=BG)
        _style(ax); return fig, ax
    fig, axes = plt.subplots(1, cols, figsize=(w, h), facecolor=BG)
    [_style(ax) for ax in axes]; return fig, axes

def _style(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=11)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.xaxis.label.set_color(WHITE)
    ax.yaxis.label.set_color(WHITE)
    ax.title.set_color(WHITE)
    ax.grid(color=GRID, lw=0.6, alpha=0.8)

print('Ready.')

---
## 📊 Chapter 1 — OPL Squat Data

In [ ]:
OPL_PATH = 'opl_data.csv'
if not os.path.exists(OPL_PATH):
    print('Downloading OPL dataset...')
    url = 'https://openpowerlifting.gitlab.io/opl-csv/files/openpowerlifting-latest.zip'
    r   = requests.get(url, stream=True, timeout=180)
    chunks = []
    for chunk in r.iter_content(chunk_size=1024*256): chunks.append(chunk)
    with zipfile.ZipFile(io.BytesIO(b''.join(chunks))) as z:
        csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
        with z.open(csv_name) as f:
            df_raw = pd.read_csv(f, low_memory=False)
    df_raw.to_csv(OPL_PATH, index=False)
else:
    df_raw = pd.read_csv(OPL_PATH, low_memory=False)
    print(f'Loaded — {len(df_raw):,} records')

In [ ]:
df = df_raw[['Sex','Best3SquatKg','Equipment']].copy()
df = df[df['Best3SquatKg'] > 0].dropna()
df['squat_lbs'] = df['Best3SquatKg'] * 2.20462
df = df[df['Equipment'].isin(['Raw','Wraps'])]
df = df[df['Sex'].isin(['M','F'])]
df = df[(df['squat_lbs'] >= 45) & (df['squat_lbs'] <= 950)]

male_opl   = df[df['Sex']=='M']['squat_lbs'].values
female_opl = df[df['Sex']=='F']['squat_lbs'].values

print(f'Clean squat records:  Male {len(male_opl):,}  |  Female {len(female_opl):,}')
print(f'Male   — median {np.median(male_opl):.1f} lbs')
print(f'Female — median {np.median(female_opl):.1f} lbs')

In [ ]:
m_shape, _, m_scale = _lognorm.fit(male_opl,   floc=0)
f_shape, _, f_scale = _lognorm.fit(female_opl, floc=0)
print(f'Male   σ={m_shape:.4f}  fit_median={m_scale:.1f} lbs')
print(f'Female σ={f_shape:.4f}  fit_median={f_scale:.1f} lbs')

x = np.linspace(45, 850, 600)
fig, axes = make_fig(13, 5, cols=2)
for ax, vals, sh, sc, color, label in [
    (axes[0], male_opl,   m_shape, m_scale, MALE, 'Male'),
    (axes[1], female_opl, f_shape, f_scale, FEM,  'Female'),
]:
    ax.hist(vals, bins=150, density=True, color=color, alpha=0.35, edgecolor='none')
    ax.plot(x, _lognorm.pdf(x, sh, scale=sc), color=WHITE, lw=2.2, label='Log-normal fit')
    ax.axvline(np.median(vals), color=color, lw=1.4, ls='--', alpha=0.8,
               label=f'Median {np.median(vals):.0f} lbs')
    ax.set_title(f'{label} — Raw OPL Squat', color=WHITE, fontsize=12, fontweight='bold')
    ax.set_xlabel('Squat (lbs)'); ax.set_ylabel('Density')
    ax.set_xlim(45, 850)
    ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
    for t, c in zip(MILESTONES, M_COLORS):
        ax.axvline(t, color=c, lw=1.2, ls=':', alpha=0.7)
fig.suptitle('Log-Normal Fit — OPL Raw Squat Data', color=WHITE, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_opl_fit.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 🏋️ Chapter 2 — Two Pools of Squatters

The bench press is almost purely a gym movement. The squat is not.

**Pool A — Gym barbell squatters:** people who specifically train the back squat in a gym setting. Same methodology as bench model: keep OPL shape (sigma), shift median down to normative gym-goer data.

**Pool B — Non-gym strength athletes:** people who have developed real leg strength through:
- Military service (20M+ active duty globally; many programs include loaded carries, ruck marching, sprinting — compound leg work)
- Combat sports — wrestling, judo, boxing, MMA, sambo (~30–50M serious practitioners)
- Rugby union/league (~8M registered players), American/Canadian football (~5M)
- Serious team sport athletes with structured S&C (professional/semi-pro soccer, basketball)

**Key adjustment for Pool B:** a 70% barbell efficiency factor. Someone who has never specifically trained the back squat movement (bracing, bar path, depth under load) can't fully express their leg strength on day one. Conservative: effective 1RM ≈ 70% of their true strength potential. This is already baked into Pool B's median — they're calibrated to what they could actually lift on a barbell, not their theoretical leg strength ceiling.

Pool B is **heavily male-dominated** (~75% male) because military and combat sports participation skews heavily male.

In [ ]:
# Visualize the two pool distributions vs OPL competitors
fig, ax = make_fig(13, 5)
x = np.linspace(20, 850, 800)

# Pool A: gym squatters (North America baseline)
gym_m_med, gym_f_med = 205.0, 115.0
gym_sig_m, gym_sig_f = 0.33, 0.345

# Pool B: non-gym strength athletes (barbell-efficiency-adjusted medians)
# Male: ~175 lbs effective squat (strong from sport, but not squat-specific trained)
# Female: ~105 lbs effective squat
ng_m_med, ng_f_med = 175.0, 105.0
ng_sig_m, ng_sig_f = 0.35, 0.36   # slightly wider — less systematic training

curves = [
    (np.median(male_opl),   m_shape,   MALE,          'Male — OPL competitors',         '-',  3.0),
    (gym_m_med,             gym_sig_m, POOL_A_COLOR,  'Male — Pool A (gym squatter)',   '--', 2.0),
    (ng_m_med,              ng_sig_m,  POOL_B_COLOR,  'Male — Pool B (non-gym athlete)',':', 2.0),
    (np.median(female_opl), f_shape,   FEM,           'Female — OPL competitors',       '-',  2.0),
    (gym_f_med,             gym_sig_f, '#c8a0d8',     'Female — Pool A',                '--', 1.5),
    (ng_f_med,              ng_sig_f,  '#ffcc80',     'Female — Pool B',                ':',  1.5),
]
for med, sigma, color, label, ls, lw in curves:
    ax.plot(x, _lognorm.pdf(x, sigma, scale=med),
            color=color, lw=lw, ls=ls, label=label, alpha=0.9)

for t, c in zip(MILESTONES, M_COLORS):
    ax.axvline(t, color=c, lw=1.2, ls=':', alpha=0.6)
    ax.text(t+4, ax.get_ylim()[1]*0.03, f'{t}', color=c, fontsize=9)

ax.set_xlabel('Squat (lbs)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Three Populations: OPL Competitors | Gym Squatters | Non-Gym Athletes',
             color=WHITE, fontsize=13, fontweight='bold')
ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID, ncol=2)
ax.set_xlim(20, 750)
plt.tight_layout()
plt.savefig('02_pools.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 🌍 Chapter 3 — Regional Parameters

**Pool A** follows the same 10-region structure as the bench model.  
**Pool B** (non-gym athletes) is defined separately — sizes derived from:
- IISS Military Balance 2023 (active duty by region)
- World combat sports federation membership estimates
- World Rugby, FIFA, FIBA registered serious-level athlete estimates

Pool B is **not** dependent on gym access — military and combat sports exist at roughly the same absolute numbers regardless of commercial gym penetration.

In [ ]:
# ── POOL A: Gym barbell squatters ───────────────────────────────────────────
# MSquatFrac: fraction of resistance trainers who back squat (lower than bench;
# many use Smith machine, leg press, or skip legs)
# FSquatFrac: women's squat rack adoption higher than bench, but still < male rate
#
# Columns: name, pop_16_70(M), lift_mode, lift_lo, lift_hi,
#          MSquatFrac, FSquatFrac,
#          male_med, male_lo, male_hi, female_med, female_lo, female_hi

REGIONS_A = [
    ('North America',         210, 0.22, 0.17, 0.28, 0.62, 0.38, 205, 185, 225, 115,  98, 132),
    ('Europe',                450, 0.15, 0.10, 0.20, 0.58, 0.34, 190, 172, 208, 108,  92, 124),
    ('Russia / C. Asia',      165, 0.10, 0.07, 0.14, 0.60, 0.34, 185, 167, 203, 105,  89, 121),
    ('East Asia',             950, 0.07, 0.04, 0.11, 0.50, 0.30, 160, 144, 176,  90,  76, 104),
    ('Latin America',         380, 0.08, 0.05, 0.12, 0.58, 0.36, 165, 149, 181,  93,  79, 107),
    ('Middle East / N. Af.',  310, 0.05, 0.03, 0.08, 0.54, 0.28, 165, 149, 181,  88,  74, 102),
    ('South Asia',           1080, 0.025,0.015,0.040,0.50, 0.28, 140, 125, 155,  77,  65,  89),
    ('SE Asia',               500, 0.035,0.020,0.055,0.52, 0.30, 148, 133, 163,  83,  70,  96),
    ('Sub-Saharan Africa',    610, 0.015,0.008,0.025,0.50, 0.28, 148, 133, 163,  83,  70,  96),
    ('Oceania',                19, 0.20, 0.16, 0.25, 0.60, 0.36, 198, 178, 218, 112,  95, 129),
]

cols_a = ['Region','Pop_M','LiftMode','LiftLo','LiftHi','MSquatFrac','FSquatFrac',
          'MMed','MMedLo','MMedHi','FMed','FMedLo','FMedHi']
df_a = pd.DataFrame(REGIONS_A, columns=cols_a)
df_a['Pop_M'] *= 1e6

df_a['EffSF'] = FRAC_MALE * df_a['MSquatFrac'] + (1-FRAC_MALE) * df_a['FSquatFrac']
df_a['Squatters_M'] = df_a['Pop_M'] * df_a['LiftMode'] * df_a['EffSF'] / 1e6
print(f'Pool A — total gym barbell squatters: {df_a["Squatters_M"].sum():.0f}M')

In [ ]:
# ── POOL B: Non-gym strength athletes ───────────────────────────────────────
#
# Population sources:
#   Military: IISS Military Balance 2023, active duty only
#   Combat sports: FILA/UWW (wrestling), IJF (judo), WBC/WBA/IBF (boxing),
#                  ONE Championship estimates, ADCC, UFC etc.
#   Rugby: World Rugby 2022 census
#   American/Canadian football: NFL/CFL pipeline + college
#   Team sports S&C: serious-level athletes with structured strength programs
#
# MaleFrac: fraction of Pool B that is male (~0.72-0.80; military+combat sports heavily male)
#
# Medians already incorporate 70% barbell efficiency factor (strength is real;
# movement pattern on day 1 of barbell squatting is not fully efficient)
#
# Columns: name, total_pool_B(M), male_frac,
#          male_med, male_lo, male_hi, female_med, female_lo, female_hi
# Sigma (shared): 0.35 male, 0.37 female — wider than gym-goers, less systematic training

REGIONS_B = [
    # name                   pool_M  mfrac  mm   mlo  mhi  fm  flo  fhi
    ('North America',          10.0, 0.74, 185, 165, 205, 112,  95, 129),
    ('Europe',                 15.0, 0.73, 175, 157, 193, 107,  90, 124),
    ('Russia / C. Asia',        8.0, 0.78, 172, 154, 190, 103,  87, 119),
    ('East Asia',              15.0, 0.79, 148, 132, 164,  90,  76, 104),
    ('Latin America',           8.0, 0.75, 152, 136, 168,  93,  79, 107),
    ('Middle East / N. Af.',    6.0, 0.82, 152, 136, 168,  88,  74, 102),
    ('South Asia',              5.0, 0.80, 130, 115, 145,  80,  67,  93),
    ('SE Asia',                 5.0, 0.80, 137, 122, 152,  83,  70,  96),
    ('Sub-Saharan Africa',      4.0, 0.79, 137, 122, 152,  83,  70,  96),
    ('Oceania',                 2.0, 0.71, 180, 161, 199, 108,  91, 125),
]

cols_b = ['Region','PoolB_M','MaleFrac',
          'MMed','MMedLo','MMedHi','FMed','FMedLo','FMedHi']
df_b = pd.DataFrame(REGIONS_B, columns=cols_b)
df_b['PoolB_M'] *= 1e6

total_b = df_b['PoolB_M'].sum() / 1e6
print(f'Pool B — total non-gym strength athletes: {total_b:.0f}M')
print(f'Breakdown: military ~20M, combat sports ~38M, rugby/football ~15M, other S&C ~5M')
print(f'Male fraction weighted avg: {(df_b["PoolB_M"]*df_b["MaleFrac"]).sum()/df_b["PoolB_M"].sum():.2f}')

---
## 🎲 Chapter 4 — Monte Carlo: Both Pools Combined

In [ ]:
N_SIM = 100_000
R     = len(df_a)
rng   = np.random.default_rng(42)

pop_a  = df_a['Pop_M'].values
msf_v  = df_a['MSquatFrac'].values
fsf_v  = df_a['FSquatFrac'].values
pop_b  = df_b['PoolB_M'].values
mfrac_b = df_b['MaleFrac'].values

# Pool A samples
lift_s   = rng.triangular(df_a['LiftLo'].values, df_a['LiftMode'].values,
                           df_a['LiftHi'].values, size=(N_SIM, R))
msf_s    = rng.uniform(msf_v * 0.85, msf_v * 1.15, size=(N_SIM, R))
fsf_s    = rng.uniform(fsf_v * 0.85, fsf_v * 1.15, size=(N_SIM, R))
a_mmed_s = rng.uniform(df_a['MMedLo'].values, df_a['MMedHi'].values, size=(N_SIM, R))
a_fmed_s = rng.uniform(df_a['FMedLo'].values, df_a['FMedHi'].values, size=(N_SIM, R))
a_msig_s = rng.uniform(0.28, 0.38, size=(N_SIM, 1))
a_fsig_s = rng.uniform(0.29, 0.40, size=(N_SIM, 1))

# Pool B samples
b_mmed_s = rng.uniform(df_b['MMedLo'].values, df_b['MMedHi'].values, size=(N_SIM, R))
b_fmed_s = rng.uniform(df_b['FMedLo'].values, df_b['FMedHi'].values, size=(N_SIM, R))
b_msig_s = rng.uniform(0.32, 0.40, size=(N_SIM, 1))  # wider — less systematic training
b_fsig_s = rng.uniform(0.33, 0.42, size=(N_SIM, 1))
# Pool B size uncertainty: ±25% (population estimates are rougher)
b_scale_s = rng.uniform(0.75, 1.25, size=(N_SIM, R))

# Pool A counts: gym lifters × squat fraction × sex split
a_mcnt = pop_a * lift_s * msf_s * FRAC_MALE         # (N_SIM, R)
a_fcnt = pop_a * lift_s * fsf_s * (1 - FRAC_MALE)

# Pool B counts: non-gym athletes × sex split (already built into MaleFrac)
b_mcnt = pop_b * mfrac_b * b_scale_s                # (N_SIM, R)
b_fcnt = pop_b * (1 - mfrac_b) * b_scale_s

mc_results_a = {}
mc_results_b = {}
mc_results   = {}  # combined

for target in MILESTONES:
    # Pool A
    pm_a = stats.norm.sf((np.log(target) - np.log(a_mmed_s)) / a_msig_s)
    pf_a = stats.norm.sf((np.log(target) - np.log(a_fmed_s)) / a_fsig_s)
    pool_a_total = (a_mcnt * pm_a + a_fcnt * pf_a).sum(axis=1) / 1e6
    mc_results_a[target] = pool_a_total

    # Pool B
    pm_b = stats.norm.sf((np.log(target) - np.log(b_mmed_s)) / b_msig_s)
    pf_b = stats.norm.sf((np.log(target) - np.log(b_fmed_s)) / b_fsig_s)
    pool_b_total = (b_mcnt * pm_b + b_fcnt * pf_b).sum(axis=1) / 1e6
    mc_results_b[target] = pool_b_total

    mc_results[target] = pool_a_total + pool_b_total

print(f'Simulation complete — {N_SIM:,} trials\n')
print(f'{"Target":<8} {"Pool A":>10} {"Pool B":>10} {"Combined":>12} {"% World":>10}')
print('-'*55)
for t in MILESTONES:
    va  = np.median(mc_results_a[t])
    vb  = np.median(mc_results_b[t])
    vc  = np.median(mc_results[t])
    lo  = np.percentile(mc_results[t], 2.5)
    hi  = np.percentile(mc_results[t], 97.5)
    pct = vc / (WORLD_POP / 1e6) * 100
    print(f'{t} lbs   {va:>8.2f}M   {vb:>8.2f}M   {vc:>8.2f}M   {pct:.5f}%')
    print(f'         95% CI: [{lo:.2f}M – {hi:.2f}M]')

---
## 👥 Chapter 5 — Denominator Analysis

The denominator you choose changes the story dramatically:

| Denominator | Who's in it | Why use it |
|---|---|---|
| All 8.2B humans | Every living person | Most honest: the question is 'how many people on Earth' |
| Able adults 16–70 | Excluded: true children, elderly 70+ | Removes people for whom it's physically impossible |
| Men 16–70 only | ~2.37B | Most relevant if asking 'how rare among men' |
| Women 16–70 only | ~2.30B | Same for women — paints a completely different picture |

In [ ]:
# Denominator analysis + male/female split
# For male/female split we need separate MC results

mc_male   = {}  # men only (both pools)
mc_female = {}  # women only (both pools)

for target in MILESTONES:
    pm_a = stats.norm.sf((np.log(target) - np.log(a_mmed_s)) / a_msig_s)
    pf_a = stats.norm.sf((np.log(target) - np.log(a_fmed_s)) / a_fsig_s)
    pm_b = stats.norm.sf((np.log(target) - np.log(b_mmed_s)) / b_msig_s)
    pf_b = stats.norm.sf((np.log(target) - np.log(b_fmed_s)) / b_fsig_s)

    mc_male[target]   = (a_mcnt*pm_a + b_mcnt*pm_b).sum(axis=1) / 1e6
    mc_female[target] = (a_fcnt*pf_a + b_fcnt*pf_b).sum(axis=1) / 1e6

print('='*72)
print(f'{"":12} {"All humans":>12} {"Able 16-70":>12} {"Men 16-70":>12} {"Women 16-70":>12}')
print(f'{"":12} {"(8.2B)":>12} {"(4.67B)":>12} {"(2.37B)":>12} {"(2.30B)":>12}')
print('='*72)
for t in MILESTONES:
    vc  = np.median(mc_results[t])
    vm  = np.median(mc_male[t])
    vf  = np.median(mc_female[t])
    print(f'\n{t} lbs  ≈{vc:.2f}M people total')
    print(f'  % of all humans:    {vc/(WORLD_POP/1e6)*100:.5f}%   (1 in {WORLD_POP/(vc*1e6):,.0f})')
    print(f'  % of able adults:   {vc/(POP_ABLE_1670/1e6)*100:.4f}%   (1 in {POP_ABLE_1670/(vc*1e6):,.0f})')
    print(f'  % of men 16-70:     {vm/(POP_MEN_1670/1e6)*100:.4f}%   (1 in {POP_MEN_1670/(vm*1e6):,.0f})')
    print(f'  % of women 16-70:   {vf/(POP_WOMEN_1670/1e6)*100:.5f}%   (1 in {POP_WOMEN_1670/(vf*1e6):,.0f})')

In [ ]:
# ── Pool A vs B stacked bar ────────────────────────────────────────────────
fig, ax = make_fig(10, 5)

x    = np.arange(len(MILESTONES))
va   = [np.median(mc_results_a[t]) for t in MILESTONES]
vb   = [np.median(mc_results_b[t]) for t in MILESTONES]
vc   = [np.median(mc_results[t])   for t in MILESTONES]

bars_a = ax.bar(x, va, color=POOL_A_COLOR, alpha=0.85, label='Pool A — gym squatters')
bars_b = ax.bar(x, vb, bottom=va, color=POOL_B_COLOR, alpha=0.85, label='Pool B — non-gym athletes')

for i, (a, b, c) in enumerate(zip(va, vb, vc)):
    ax.text(i, c + 0.05, f'{c:.2f}M total', ha='center', color=WHITE, fontsize=11, fontweight='bold')
    ax.text(i, a/2,     f'{a:.2f}M', ha='center', color=BG,   fontsize=10)
    ax.text(i, a + b/2, f'{b:.2f}M', ha='center', color=BG,   fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(M_TAGS, color=WHITE, fontsize=12)
ax.set_ylabel('People worldwide (millions)', fontsize=11)
ax.set_title('Pool A (gym) vs Pool B (non-gym athletes) — Both Pools Combined',
             color=WHITE, fontsize=13, fontweight='bold')
ax.legend(fontsize=10, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
plt.tight_layout()
plt.savefig('03_pool_split.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Male vs Female split ──────────────────────────────────────────────────
fig, ax = make_fig(10, 5)
x  = np.arange(len(MILESTONES))
vm = [np.median(mc_male[t])   for t in MILESTONES]
vf = [np.median(mc_female[t]) for t in MILESTONES]

w = 0.35
bm = ax.bar(x - w/2, vm, w, color=MALE, alpha=0.85, label='Male')
bf = ax.bar(x + w/2, vf, w, color=FEM,  alpha=0.85, label='Female')

for i, (m, f) in enumerate(zip(vm, vf)):
    pct_m = m / (POP_MEN_1670/1e6) * 100
    pct_f = f / (POP_WOMEN_1670/1e6) * 100
    ax.text(i-w/2, m+0.03, f'{m:.2f}M\n({pct_m:.3f}%)',
            ha='center', va='bottom', color=MALE, fontsize=9, fontweight='bold')
    ax.text(i+w/2, f+0.03, f'{f:.2f}M\n({pct_f:.4f}%)',
            ha='center', va='bottom', color=FEM,  fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(M_TAGS, color=WHITE, fontsize=12)
ax.set_ylabel('People worldwide (millions)', fontsize=11)
ax.set_title('Male vs Female — Who Can Actually Hit These Numbers?',
             color=WHITE, fontsize=13, fontweight='bold')
ax.legend(fontsize=11, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
plt.tight_layout()
plt.savefig('04_male_female_split.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Full drop-off curve: 100-600 lbs ────────────────────────────────────────
X_TARGETS = np.linspace(100, 600, 200)
N_CURVE   = 10_000
rng3      = np.random.default_rng(7)

# Pool A curve samples
lc_a   = rng3.triangular(df_a['LiftLo'].values, df_a['LiftMode'].values,
                          df_a['LiftHi'].values, size=(N_CURVE, R))
mc_as  = rng3.uniform(msf_v*0.85, msf_v*1.15, size=(N_CURVE, R))
fc_as  = rng3.uniform(fsf_v*0.85, fsf_v*1.15, size=(N_CURVE, R))
am_mc  = rng3.uniform(df_a['MMedLo'].values, df_a['MMedHi'].values, size=(N_CURVE, R))
af_mc  = rng3.uniform(df_a['FMedLo'].values, df_a['FMedHi'].values, size=(N_CURVE, R))
ams_c  = rng3.uniform(0.28, 0.38, size=(N_CURVE,1,1))
afs_c  = rng3.uniform(0.29, 0.40, size=(N_CURVE,1,1))

a_mc3  = (pop_a * lc_a * mc_as * FRAC_MALE)[:,:,None]
a_fc3  = (pop_a * lc_a * fc_as * (1-FRAC_MALE))[:,:,None]

# Pool B curve samples
bsc    = rng3.uniform(0.75, 1.25, size=(N_CURVE, R))
bm_mc  = rng3.uniform(df_b['MMedLo'].values, df_b['MMedHi'].values, size=(N_CURVE, R))
bf_mc  = rng3.uniform(df_b['FMedLo'].values, df_b['FMedHi'].values, size=(N_CURVE, R))
bms_c  = rng3.uniform(0.32, 0.40, size=(N_CURVE,1,1))
bfs_c  = rng3.uniform(0.33, 0.42, size=(N_CURVE,1,1))
b_mc3  = (pop_b * mfrac_b * bsc)[:,:,None]
b_fc3  = (pop_b * (1-mfrac_b) * bsc)[:,:,None]

ln_x   = np.log(X_TARGETS)[None,None,:]

curve_a = (a_mc3 * stats.norm.sf((ln_x - np.log(am_mc[:,:,None])) / ams_c) +
           a_fc3 * stats.norm.sf((ln_x - np.log(af_mc[:,:,None])) / afs_c)).sum(axis=1)/1e6
curve_b = (b_mc3 * stats.norm.sf((ln_x - np.log(bm_mc[:,:,None])) / bms_c) +
           b_fc3 * stats.norm.sf((ln_x - np.log(bf_mc[:,:,None])) / bfs_c)).sum(axis=1)/1e6
curve_t = curve_a + curve_b

c_med  = np.median(curve_t, axis=0)
c_lo   = np.percentile(curve_t, 2.5,  axis=0)
c_hi   = np.percentile(curve_t, 97.5, axis=0)
ca_med = np.median(curve_a, axis=0)

fig, ax = make_fig(13, 6)
ax.fill_between(X_TARGETS, c_lo, c_hi, color=WHITE, alpha=0.10, label='95% CI (combined)')
ax.plot(X_TARGETS, c_med,  color=WHITE,        lw=2.5, label='Combined (A+B)')
ax.plot(X_TARGETS, ca_med, color=POOL_A_COLOR, lw=1.5, ls='--', alpha=0.7, label='Pool A only (gym)')

for t, color, lbl in zip(MILESTONES, M_COLORS, M_LABELS):
    idx = int(np.argmin(np.abs(X_TARGETS - t)))
    ax.axvline(t, color=color, lw=1.5, ls=':', alpha=0.7)
    ax.scatter([t], [c_med[idx]], color=color, s=90, zorder=5)
    ax.text(t+5, c_med[idx]*1.4, f'{lbl}\n{c_med[idx]:.2f}M',
            color=color, fontsize=9.5, va='bottom')

ax.set_yscale('log')
ax.set_xlabel('Squat (lbs)', fontsize=12)
ax.set_ylabel('People worldwide (millions, log)', fontsize=12)
ax.set_title('Global Squat Capability — Full Drop-Off Curve (Both Pools)',
             color=WHITE, fontsize=14, fontweight='bold')
ax.legend(fontsize=10, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.3g}M'))
plt.tight_layout()
plt.savefig('05_dropoff.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 🎬 IG Reel Slides — 9:16 Portrait

In [ ]:
# Shared weighted medians for the distribution illustration
_mw = df_a['Pop_M'] * df_a['LiftMode'] * df_a['MSquatFrac']
_fw = df_a['Pop_M'] * df_a['LiftMode'] * df_a['FSquatFrac']
_gm_med = float(np.average(df_a['MMed'], weights=_mw))
_gf_med = float(np.average(df_a['FMed'], weights=_fw))
_m_sig, _f_sig = 0.33, 0.345

def reveal_reel(target, color, ax, extra_note=''):
    sims   = mc_results[target]
    med    = np.median(sims)
    lo, hi = np.percentile(sims, [2.5, 97.5])
    pct_all  = med / (WORLD_POP/1e6) * 100
    pct_able = med / (POP_ABLE_1670/1e6) * 100
    vm  = np.median(mc_male[target])
    vf  = np.median(mc_female[target])

    x = np.linspace(20, 750, 1200)
    pdf_m = _lognorm.pdf(x, _m_sig, scale=_gm_med)
    pdf_f = _lognorm.pdf(x, _f_sig, scale=_gf_med)

    ax.plot(x, pdf_m, color=MALE, lw=2.5, alpha=0.85)
    ax.plot(x, pdf_f, color=FEM,  lw=2.5, alpha=0.85)
    mask = x >= target
    ax.fill_between(x[mask], pdf_m[mask], color=MALE, alpha=0.30)
    ax.fill_between(x[mask], pdf_f[mask], color=FEM,  alpha=0.30)
    ax.axvline(target, color=color, lw=3.0, ls='--', alpha=0.95)

    y_top = max(pdf_m.max(), pdf_f.max())
    ax.text(target+8, y_top*0.92, f'{target} lbs',
            color=color, fontsize=13, fontweight='bold', va='top')

    # Main number
    ax.text(0.5, 0.56, f'≈ {med:.2f}M people',
            transform=ax.transAxes, ha='center', va='center',
            color=color, fontsize=21, fontweight='black')
    # Denominator lines
    ax.text(0.5, 0.46,
            f'{pct_all:.4f}% of all humans\n{pct_able:.3f}% of able-bodied adults',
            transform=ax.transAxes, ha='center', va='center',
            color=WHITE, fontsize=12, fontweight='bold')
    # M/F split
    ax.text(0.5, 0.33,
            f'Men: {vm:.2f}M  |  Women: {vf:.2f}M',
            transform=ax.transAxes, ha='center', va='center',
            color=GRAY, fontsize=11)
    ax.text(0.5, 0.26, f'CI: {lo:.2f}M – {hi:.2f}M',
            transform=ax.transAxes, ha='center', va='center',
            color=GRAY, fontsize=10)
    ax.set_xlim(20, 750)
    ax.set_xlabel('Squat (lbs)', fontsize=12)

In [ ]:
# ── Slide 00 — HOOK ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
ax  = fig.add_axes([0, 0, 1, 1]); ax.set_facecolor(BG); ax.axis('off')
T   = ax.transAxes

ax.axhline(0.93, color=C315, lw=5, xmin=0.07, xmax=0.93)

ax.text(0.5, 0.87, 'How many people', ha='center', va='top',
        color=WHITE, fontsize=34, fontweight='black', transform=T)
ax.text(0.5, 0.79, 'on Earth can', ha='center', va='top',
        color=WHITE, fontsize=34, fontweight='black', transform=T)
ax.text(0.5, 0.70, 'SQUAT', ha='center', va='top',
        color=C315, fontsize=62, fontweight='black', transform=T)
ax.text(0.5, 0.60, '3 plates?', ha='center', va='top',
        color=WHITE, fontsize=46, fontweight='black', transform=T)

ax.axhline(0.55, color=GRID, lw=1.5, xmin=0.1, xmax=0.9, alpha=0.5)
ax.text(0.5, 0.51, 'Not just gym rats.', ha='center', va='top',
        color=WHITE, fontsize=20, fontweight='bold', transform=T)
ax.text(0.5, 0.45, 'Military. Combat sports. Athletes.\nEveryone on Earth — 8.2 billion.',
        ha='center', va='top', color=GRAY, fontsize=16, transform=T)
ax.text(0.5, 0.36, 'I built a model to find out.', ha='center', va='top',
        color=WHITE, fontsize=18, fontweight='bold', transform=T)

ax.axhline(0.30, color=GRID, lw=1.0, xmin=0.1, xmax=0.9, alpha=0.4)
for i, (m, c, lbl) in enumerate(zip(MILESTONES, M_COLORS, ['3 plates','4 plates','5 plates'])):
    x_pos = 0.18 + i*0.32
    ax.text(x_pos, 0.26, f'{m}', ha='center', color=c, fontsize=28, fontweight='black', transform=T)
    ax.text(x_pos, 0.20, lbl,    ha='center', color=GRAY, fontsize=13, transform=T)

ax.text(0.5, 0.10, 'The answer is humbling.\nStay for 495 →', ha='center', va='top',
        color=C495, fontsize=16, fontweight='bold', transform=T)
ax.axhline(0.04, color=C495, lw=3, xmin=0.08, xmax=0.92, alpha=0.6)

plt.savefig('reel_00_hook.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 00 saved')

In [ ]:
# ── Slide 01 — 315 lbs ────────────────────────────────────────────────────
v315 = np.median(mc_results[315])
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
fig.text(0.5, 0.97, '3 PLATES', ha='center', va='top',
         color=C315, fontsize=50, fontweight='black')
fig.text(0.5, 0.89, '315 lbs — every human counted', ha='center', va='top',
         color=GRAY, fontsize=15)

ax = fig.add_axes([0.07, 0.30, 0.88, 0.55])
_style(ax); reveal_reel(315, C315, ax)

fig.text(0.5, 0.27, f'Gym squatters + military + combat sports.', ha='center', va='top',
         color=GRAY, fontsize=13)
fig.text(0.5, 0.22, f'Still only ≈{v315:.1f}M people.\nOut of 8.2 billion.', ha='center', va='top',
         color=WHITE, fontsize=16, fontweight='bold')
fig.text(0.5, 0.10, '↓ 405 next ↓', ha='center', va='top',
         color=C405, fontsize=17, fontweight='bold')

plt.savefig('reel_01_315.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print(f'Slide 01 — 315 lbs: ≈{v315:.2f}M')

In [ ]:
# ── Slide 02 — 405 lbs ────────────────────────────────────────────────────
v405 = np.median(mc_results[405])
drop = v405 / v315 * 100
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
fig.text(0.5, 0.97, '4 PLATES', ha='center', va='top',
         color=C405, fontsize=50, fontweight='black')
fig.text(0.5, 0.89, '405 lbs — serious territory', ha='center', va='top',
         color=GRAY, fontsize=15)

ax = fig.add_axes([0.07, 0.30, 0.88, 0.55])
_style(ax); reveal_reel(405, C405, ax)

fig.text(0.5, 0.27, f'From 315 to 405 — the pool shrinks by {100-drop:.0f}%.', ha='center',
         va='top', color=GRAY, fontsize=13)
fig.text(0.5, 0.21, f'≈{v405:.2f}M people left.', ha='center', va='top',
         color=WHITE, fontsize=17, fontweight='bold')
fig.text(0.5, 0.10, '↓ 495 changes everything ↓', ha='center', va='top',
         color=C495, fontsize=17, fontweight='bold')

plt.savefig('reel_02_405.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print(f'Slide 02 — 405 lbs: ≈{v405:.2f}M')

In [ ]:
# ── Slide 03 — 495 lbs ────────────────────────────────────────────────────
v495 = np.median(mc_results[495])
in_x_men = int(POP_MEN_1670 / (np.median(mc_male[495])*1e6))
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
fig.text(0.5, 0.97, '5 PLATES', ha='center', va='top',
         color=C495, fontsize=50, fontweight='black')
fig.text(0.5, 0.89, '495 lbs — most lifters never see this', ha='center', va='top',
         color=GRAY, fontsize=15)

ax = fig.add_axes([0.07, 0.30, 0.88, 0.55])
_style(ax); reveal_reel(495, C495, ax)

fig.text(0.5, 0.27, f'1 in every {in_x_men:,} men on Earth.', ha='center', va='top',
         color=C495, fontsize=16, fontweight='bold')
fig.text(0.5, 0.21, 'That\'s not a gym milestone.\nThat\'s a sports stadium.', ha='center', va='top',
         color=WHITE, fontsize=15)
fig.text(0.5, 0.10, '↓ See the full picture ↓', ha='center', va='top',
         color=WHITE, fontsize=17, fontweight='bold')

plt.savefig('reel_03_495.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print(f'Slide 03 — 495 lbs: ≈{v495:.2f}M')

In [ ]:
# ── Slide 04 — Drop-off curve (portrait) ─────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
fig.text(0.5, 0.97, 'THE DROP-OFF', ha='center', va='top',
         color=WHITE, fontsize=36, fontweight='black')
fig.text(0.5, 0.91, 'Every person who could possibly squat',
         ha='center', va='top', color=GRAY, fontsize=15)

ax = fig.add_axes([0.11, 0.22, 0.85, 0.65])
_style(ax)
ax.fill_between(X_TARGETS, c_lo, c_hi, color=WHITE, alpha=0.08)
ax.plot(X_TARGETS, c_med,  color=WHITE,        lw=2.5, label='Combined (gym + athletes)')
ax.plot(X_TARGETS, ca_med, color=POOL_A_COLOR, lw=1.5, ls='--', alpha=0.6, label='Gym only')

for t, color, lbl in zip(MILESTONES, M_COLORS, M_LABELS):
    idx = int(np.argmin(np.abs(X_TARGETS - t)))
    ax.axvline(t, color=color, lw=2.0, ls=':', alpha=0.8)
    ax.scatter([t], [c_med[idx]], color=color, s=90, zorder=5)
    ax.text(t+5, c_med[idx]*1.6, f'{lbl}\n{c_med[idx]:.2f}M',
            color=color, fontsize=9, va='bottom')

ax.set_yscale('log')
ax.set_xlabel('Squat (lbs)', fontsize=13)
ax.set_ylabel('People worldwide (M, log)', fontsize=12)
ax.legend(fontsize=10, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.3g}M'))

fig.text(0.5, 0.18, 'Every 90 lbs = ~85% fewer people.',
         ha='center', va='top', color=WHITE, fontsize=15, fontweight='bold')
fig.text(0.5, 0.13, 'Non-gym athletes add meaningful numbers\nat 315 — but vanish at 495.',
         ha='center', va='top', color=GRAY, fontsize=13)

plt.savefig('reel_04_dropoff.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 04 — Drop-off saved')

In [ ]:
# ── Slide 05 — Denominator frame (the perspective slide) ────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
ax  = fig.add_axes([0, 0, 1, 1]); ax.set_facecolor(BG); ax.axis('off'); T = ax.transAxes

ax.text(0.5, 0.96, 'DEPENDS WHAT YOU COMPARE TO',
        ha='center', va='top', color=WHITE, fontsize=22, fontweight='black', transform=T)
ax.axhline(0.91, color=GRID, lw=1.5, xmin=0.07, xmax=0.93, alpha=0.5)

rows = [
    ('315 lbs',   C315, [
        ('All 8.2B humans',  f'{np.median(mc_results[315])/(WORLD_POP/1e6)*100:.4f}%',
         f'1 in {int(WORLD_POP/(np.median(mc_results[315])*1e6)):,}'),
        ('Able adults 16-70', f'{np.median(mc_results[315])/(POP_ABLE_1670/1e6)*100:.3f}%',
         f'1 in {int(POP_ABLE_1670/(np.median(mc_results[315])*1e6)):,}'),
        ('Men 16-70 only',   f'{np.median(mc_male[315])/(POP_MEN_1670/1e6)*100:.3f}%',
         f'1 in {int(POP_MEN_1670/(np.median(mc_male[315])*1e6)):,}'),
    ]),
    ('405 lbs',   C405, [
        ('All 8.2B humans',  f'{np.median(mc_results[405])/(WORLD_POP/1e6)*100:.5f}%',
         f'1 in {int(WORLD_POP/(np.median(mc_results[405])*1e6)):,}'),
        ('Able adults 16-70', f'{np.median(mc_results[405])/(POP_ABLE_1670/1e6)*100:.4f}%',
         f'1 in {int(POP_ABLE_1670/(np.median(mc_results[405])*1e6)):,}'),
        ('Men 16-70 only',   f'{np.median(mc_male[405])/(POP_MEN_1670/1e6)*100:.4f}%',
         f'1 in {int(POP_MEN_1670/(np.median(mc_male[405])*1e6)):,}'),
    ]),
    ('495 lbs',   C495, [
        ('All 8.2B humans',  f'{np.median(mc_results[495])/(WORLD_POP/1e6)*100:.6f}%',
         f'1 in {int(WORLD_POP/(np.median(mc_results[495])*1e6)):,}'),
        ('Able adults 16-70', f'{np.median(mc_results[495])/(POP_ABLE_1670/1e6)*100:.5f}%',
         f'1 in {int(POP_ABLE_1670/(np.median(mc_results[495])*1e6)):,}'),
        ('Men 16-70 only',   f'{np.median(mc_male[495])/(POP_MEN_1670/1e6)*100:.5f}%',
         f'1 in {int(POP_MEN_1670/(np.median(mc_male[495])*1e6)):,}'),
    ]),
]

y = 0.88
for milestone_lbl, color, entries in rows:
    ax.text(0.07, y, milestone_lbl, ha='left', va='top',
            color=color, fontsize=20, fontweight='black', transform=T)
    y -= 0.052
    for denom_lbl, pct, one_in in entries:
        ax.text(0.09, y, f'{denom_lbl}:', ha='left', va='top',
                color=GRAY, fontsize=12, transform=T)
        ax.text(0.93, y, f'{pct}  ({one_in})', ha='right', va='top',
                color=WHITE, fontsize=12, fontweight='bold', transform=T)
        y -= 0.044
    ax.axhline(y+0.01, color=GRID, lw=0.8, xmin=0.07, xmax=0.93, alpha=0.5)
    y -= 0.012

ax.text(0.5, y-0.02, 'How you frame it is everything.',
        ha='center', va='top', color=WHITE, fontsize=16, fontweight='bold', transform=T)
ax.text(0.5, y-0.07, 'But the number is always small.', ha='center', va='top',
        color=GRAY, fontsize=14, transform=T)

plt.savefig('reel_05_denominators.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 05 — Denominator frame saved')

In [ ]:
# ── Slide 06 — Male vs Female reveal ─────────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
fig.text(0.5, 0.97, 'MEN vs WOMEN', ha='center', va='top',
         color=WHITE, fontsize=36, fontweight='black')
fig.text(0.5, 0.91, 'Very different stories', ha='center', va='top',
         color=GRAY, fontsize=16)

ax = fig.add_axes([0.08, 0.32, 0.86, 0.54])
_style(ax)
x  = np.arange(len(MILESTONES))
vm = [np.median(mc_male[t])   for t in MILESTONES]
vf = [np.median(mc_female[t]) for t in MILESTONES]
w  = 0.35
ax.bar(x-w/2, vm, w, color=MALE, alpha=0.88, label='Men')
ax.bar(x+w/2, vf, w, color=FEM,  alpha=0.88, label='Women')
for i, (m, f) in enumerate(zip(vm, vf)):
    pm = m/(POP_MEN_1670/1e6)*100
    pf = f/(POP_WOMEN_1670/1e6)*100
    ax.text(i-w/2, m+0.02, f'{m:.2f}M\n{pm:.3f}%',
            ha='center', va='bottom', color=MALE, fontsize=9.5, fontweight='bold')
    ax.text(i+w/2, f+0.02, f'{f:.3f}M\n{pf:.4f}%',
            ha='center', va='bottom', color=FEM,  fontsize=9.5, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(M_TAGS, color=WHITE, fontsize=12)
ax.set_ylabel('People worldwide (millions)', fontsize=12)
ax.legend(fontsize=12, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)

ratio_315 = vm[0]/vf[0]
fig.text(0.5, 0.29, f'Men are {ratio_315:.0f}× more likely to squat 3 plates.',
         ha='center', va='top', color=WHITE, fontsize=14, fontweight='bold')
fig.text(0.5, 0.23, 'Not because of potential —\nbecause of who trains and how.',
         ha='center', va='top', color=GRAY, fontsize=13)
fig.text(0.5, 0.10, 'What\'s your squat? Drop it below 👇',
         ha='center', va='top', color=C315, fontsize=16, fontweight='bold')

plt.savefig('reel_06_male_female.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 06 — M/F saved')

In [ ]:
# ── Slide 07 — Summary CTA ────────────────────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
ax  = fig.add_axes([0,0,1,1]); ax.set_facecolor(BG); ax.axis('off'); T = ax.transAxes

ax.axhline(0.94, color=C315, lw=5, xmin=0.07, xmax=0.93)
ax.text(0.5, 0.90, 'THE NUMBERS', ha='center', va='top',
        color=WHITE, fontsize=34, fontweight='black', transform=T)
ax.text(0.5, 0.84, '(gym squatters + military + combat sports)',
        ha='center', va='top', color=GRAY, fontsize=13, transform=T)

for i, (t, c, lbl) in enumerate(zip(MILESTONES, M_COLORS, ['3 plates','4 plates','5 plates'])):
    med   = np.median(mc_results[t])
    vm    = np.median(mc_male[t])
    vf    = np.median(mc_female[t])
    pct   = med/(WORLD_POP/1e6)*100
    ybase = 0.76 - i*0.20
    ax.text(0.07, ybase,       f'{t} lbs ({lbl})', ha='left', va='top',
            color=c, fontsize=18, fontweight='bold', transform=T)
    ax.text(0.07, ybase-0.052, f'≈ {med:.2f}M total  •  {pct:.4f}% of Earth', ha='left', va='top',
            color=WHITE, fontsize=13, transform=T)
    ax.text(0.07, ybase-0.095, f'Men: {vm:.2f}M  |  Women: {vf:.3f}M', ha='left', va='top',
            color=GRAY, fontsize=12, transform=T)
    ax.axhline(ybase-0.115, color=GRID, lw=0.8, xmin=0.07, xmax=0.93, alpha=0.5)

ax.text(0.5, 0.17, 'Model: 3.9M OPL lifts • 10 regions\n100,000 Monte Carlo trials\nGym + non-gym athletes',
        ha='center', va='top', color=GRAY, fontsize=12, transform=T)
ax.text(0.5, 0.08, 'How much do you squat?\nDrop it below 👇',
        ha='center', va='top', color=WHITE, fontsize=18, fontweight='bold', transform=T)
ax.axhline(0.03, color=C495, lw=3, xmin=0.08, xmax=0.92, alpha=0.7)

plt.savefig('reel_07_summary.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show(); print('Slide 07 — Summary saved')

In [ ]:
print('='*65)
print('SQUAT WORLD MODEL v2 — FINAL ESTIMATES')
print('Both pools: gym squatters + non-gym strength athletes')
print('='*65)
for t in MILESTONES:
    vc  = np.median(mc_results[t])
    va  = np.median(mc_results_a[t])
    vb  = np.median(mc_results_b[t])
    vm  = np.median(mc_male[t])
    vf  = np.median(mc_female[t])
    lo  = np.percentile(mc_results[t], 2.5)
    hi  = np.percentile(mc_results[t], 97.5)
    print(f'\n{t} lbs ({t//45} plates):')
    print(f'  Pool A (gym):    {va:.2f}M')
    print(f'  Pool B (athlete):{vb:.2f}M')
    print(f'  Combined:        {vc:.2f}M  [{lo:.2f}M – {hi:.2f}M]')
    print(f'  Male:            {vm:.2f}M  |  Female: {vf:.3f}M')
    print(f'  % all humans:    {vc/(WORLD_POP/1e6)*100:.5f}%')
    print(f'  % able adults:   {vc/(POP_ABLE_1670/1e6)*100:.4f}%')
    print(f'  % of men 16-70:  {vm/(POP_MEN_1670/1e6)*100:.4f}%')
    print(f'  1 in every:      {int(WORLD_POP/(vc*1e6)):,} humans  /  {int(POP_MEN_1670/(vm*1e6)):,} men')